# 混合特征与多组目标编码LightGBM

负责人：B。这是中文教学参考，正式实现由成员理解后编写、执行并核对。

输入挂载：官方比赛数据、任务00输出的固定共享Dataset、原始EV数据集中的EV_Adoption_and_Range_Anxiety_Dataset.csv。无需挂载旧track代码包。CPU训练，四线程；机器等待另计。

依据Megayak公开Hybrid配方及本项目已运行的五折适配。保留实际阈值与参数以复现历史方案；阈值属于经验规则，不能声称理论最优。


- [比赛数据与规则](https://www.kaggle.com/competitions/playground-series-s6e9)
- [LightGBM论文](https://proceedings.neurips.cc/paper/2017/hash/6449f44a102fde848669bdd9eb6b76fa-Abstract.html)
- [目标编码：内部交叉拟合与平滑](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.TargetEncoder.html)
- [AUC定义](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.roc_auc_score.html)

本教程生成时尚未执行Kaggle完整训练。历史分数是核对参照，不是本轮结果。阅读当前文档不意味着升级历史环境。

## 如何学习本文件

每次只运行一个单元，先用自己的话预测输出。`iloc`按位置取行，`loc`按标签取行；`to_numpy`去掉索引，之后必须保证位置对应。`assert`是验收条件，失败应查数据而非删除检查。`fit`从数据学习，`transform`使用已学习规则。

编程练习：修改一个小例子的输入并解释变化；正式配置保持历史定义。复杂特征组的整体增益不能归因于单一列。

## 读取官方数据

路径检查避免读错版本；对齐函数先检查ID集合，再恢复官方顺序。

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
from time import perf_counter
import json
import gc
import numpy as np
import pandas as pd
import sklearn
import lightgbm as lgb
from IPython.display import display
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import TargetEncoder
from sklearn.model_selection import StratifiedKFold

INPUT = Path('/kaggle/input')
TARGET = 'Will_Buy_EV'
SEED = 42
N_SPLITS = 5

def unique_file(name):
    """从已挂载输入中定位唯一文件；多个版本时停止，防止静默读错。"""
    paths = list(INPUT.rglob(name))
    assert len(paths) == 1, f'Expected one {name}, found {paths}'
    return paths[0]

competition_dirs = [INPUT/'competitions/playground-series-s6e9', INPUT/'playground-series-s6e9']
available = [p for p in competition_dirs if (p/'train.csv').is_file()]
assert len(available) == 1, 'Attach the official competition data.'
DATA = available[0]
train = pd.read_csv(DATA/'train.csv')
test = pd.read_csv(DATA/'test.csv')
sample = pd.read_csv(DATA/'sample_submission.csv')
y = train[TARGET].map({'No':0, 'Yes':1})
assert y.notna().all() and set(y.unique()) == {0,1}
assert train.id.is_unique and test.id.is_unique
assert sample.columns.tolist() == ['id',TARGET] and sample.id.equals(test.id)
assert train.columns.drop(['id',TARGET]).tolist() == test.columns.drop('id').tolist()

def align_rows(frame, ids):
    """先检查一一对应，再按官方顺序排列；不能直接假设CSV行序相同。"""
    assert frame.id.is_unique and len(frame) == len(ids)
    assert set(frame.id) == set(ids)
    return frame.set_index('id').loc[ids].reset_index()

def current_versions():
    return {'lightgbm':lgb.__version__, 'sklearn':sklearn.__version__,
            'numpy':np.__version__, 'pandas':pd.__version__}

## 读取共同分组

共享fold决定训练与验证行，三人不能重新划分。

In [ ]:
fold_path = unique_file('shared_folds.csv')
shared = align_rows(pd.read_csv(fold_path), train.id)
assert np.array_equal(shared.target, y)
assert shared.fold.notna().all() and shared.fold.isin(range(5)).all()
assert set(shared.fold) == set(range(5))
fold_ids = shared.fold.to_numpy(dtype=int)
foundation_note = json.loads((fold_path.parent/'dataset_note.json').read_text())
display(shared.groupby('fold').agg(rows=('id','size'),positive_rate=('target','mean')))
print(current_versions())

## 核对环境和读取原始数据

表格比较历史与当前环境。若版本不同，先在Kaggle安装历史摘要记录的版本并重启会话，再从头运行；不要静默升级。原始数据用于统计特征，训练模型仍使用比赛训练行。

In [ ]:
expected_versions = foundation_note['historical_versions']['hybrid']
normalized = {('sklearn' if k=='scikit_learn' else k):v for k,v in expected_versions.items()}
comparison = pd.DataFrame({'historical':normalized,'current':current_versions()})
display(comparison)
assert normalized, 'Historical environment metadata is missing.'
assert all(current_versions().get(k)==v for k,v in normalized.items() if k in current_versions()), 'Match historical library versions and restart the session.'
original_path = unique_file('EV_Adoption_and_Range_Anxiety_Dataset.csv')
original = pd.read_csv(original_path)
print(original.shape)

### 原始EV数据的身份与保留理由

[官方Data页面](https://www.kaggle.com/competitions/playground-series-s6e9/data)明确链接这份[Omkar Kadam发布的EV源数据](https://www.kaggle.com/datasets/itzzomkar/ev-adoption-behavior-and-range-anxiety)，措辞是比赛训练/测试数据受其启发、分布接近但不相同。源数据本身是10,000行合成记录，不能当作真实调查。

历史强模型删除12列外部均值后OOF仅下降约0.00000515、公开榜显示持平；不能宣称外部均值已有可靠私人榜收益。本次保留是为了重建已验证的完整配置。

## 准备原始数据均值

先在原始EV数据上按单个特征计算购买均值。这里不读取比赛验证标签。两种最终模型对原始数据清洗规则不同，按历史实现保留。

In [ ]:
feature_columns = [column for column in test.columns if column != "id"]
categorical_columns = [
    "Gender", "City_Type", "Current_Car_Type",
    "Home_Charging_Possible", "Subsidy_Available",
    "Range_Anxiety_Level",
]
numeric_columns = [column for column in feature_columns if column not in categorical_columns]
original_clean = original.dropna(
    subset=["Annual_Income_USD", "Daily_Commute_km", "Environmental_Concern_Level"]
).copy()
original_clean[TARGET] = original_clean[TARGET].map({"No": 0, "Yes": 1})
assert original_clean[TARGET].notna().all()
original_prior = float(original_clean[TARGET].mean())
original_means = {
    column: original_clean.groupby(column, observed=True)[TARGET].mean()
    for column in categorical_columns + numeric_columns
}

## 构造数字、分组和阈值特征

将收入取整，通勤乘10再四舍五入；用余数提取个位、十位及尾数，用整除形成多尺度分组。阈值标志来自公开配方；不要根据本轮验证标签重新修改边界。返回特征表、编码key、收入和通勤数组。

In [ ]:
def make_unsupervised_base(data):
    frame = data[feature_columns].copy().reset_index(drop=True)
    # 纯数值规则不学习比赛标签；floor与rint的差别必须保留。
    income = np.floor(frame["Annual_Income_USD"]).to_numpy(dtype=np.int64)
    commute10 = np.rint(frame["Daily_Commute_km"].to_numpy(float) * 10).astype(np.int64)
    frame["inc_d1"] = (income % 10).astype("int8")
    frame["inc_d2"] = (income // 10 % 10).astype("int8")
    frame["inc_d3"] = (income // 100 % 10).astype("int8")
    frame["inc_mod100"] = (income % 100).astype("int16")
    frame["inc_mod1000"] = (income % 1000).astype("int16")
    frame["km_d1"] = (commute10 % 10).astype("int8")
    frame["km_mod100"] = (commute10 % 100).astype("int8")
    for divisor in (50, 100, 250, 500, 1000, 2500, 5000):
        frame[f"inc_q{divisor}"] = (income // divisor).astype("int32")
    for divisor in (5, 10, 25, 50):
        frame[f"km_q{divisor}"] = (commute10 // divisor).astype("int32")
    frame["is_30k_spike"] = (income == 30000).astype("int8")
    frame["is_millionaire_cliff"] = (income >= 170537).astype("int8")
    frame["is_dead_zone"] = ((income >= 38000) & (income <= 42000)).astype("int8")
    frame["is_env_hater"] = (frame["Environmental_Concern_Level"] == 1).astype("int8")
    for column in categorical_columns + numeric_columns:
        frame[f"{column}_org_mean"] = (
            frame[column].map(original_means[column]).fillna(original_prior).astype("float32")
        )
    keys = pd.DataFrame({
        "k_inc_exact": income.astype(str),
        "k_inc100": (income // 100).astype(str),
        "k_inc1000": (income // 1000).astype(str),
        "k_km_int": (commute10 // 10).astype(str),
    })
    for column in categorical_columns + [
        "Age", "Number_of_Cars_Owned", "Charging_Stations_Near_Home",
        "Charging_Stations_Near_Work", "Environmental_Concern_Level",
    ]:
        keys[f"k_{column}"] = frame[column].astype(str).to_numpy()
    return frame, keys, income, commute10

### 对照上方代码逐句理解

行号从上方代码单元第一行起计。重复操作也列出，方便逐行定位；跨行调用请连同后续参数一起阅读。

| 行 | 代码定位 | 中文解释 |
|---|---|---|
| 4 | `income = np.floor(frame["Annual_Income_USD"]).to_numpy(dtype=np.int64)` | 向下取整建立分组；输入除以100或1000时表示更粗的收入区间。 |
| 5 | `commute10 = np.rint(frame["Daily_Commute_km"].to_numpy(float) * 10).astype(np.int64)` | 四舍五入到最近整数；与floor向下取整含义不同。 |
| 6 | `frame["inc_d1"] = (income % 10).astype("int8")` | 固定数据类型；float32节约内存，int类型表达离散分组，str把数值作为类别键。 |
| 7 | `frame["inc_d2"] = (income // 10 % 10).astype("int8")` | 固定数据类型；float32节约内存，int类型表达离散分组，str把数值作为类别键。 |
| 8 | `frame["inc_d3"] = (income // 100 % 10).astype("int8")` | 固定数据类型；float32节约内存，int类型表达离散分组，str把数值作为类别键。 |
| 9 | `frame["inc_mod100"] = (income % 100).astype("int16")` | 固定数据类型；float32节约内存，int类型表达离散分组，str把数值作为类别键。 |
| 10 | `frame["inc_mod1000"] = (income % 1000).astype("int16")` | 固定数据类型；float32节约内存，int类型表达离散分组，str把数值作为类别键。 |
| 11 | `frame["km_d1"] = (commute10 % 10).astype("int8")` | 固定数据类型；float32节约内存，int类型表达离散分组，str把数值作为类别键。 |
| 12 | `frame["km_mod100"] = (commute10 % 100).astype("int8")` | 固定数据类型；float32节约内存，int类型表达离散分组，str把数值作为类别键。 |
| 14 | `frame[f"inc_q{divisor}"] = (income // divisor).astype("int32")` | 固定数据类型；float32节约内存，int类型表达离散分组，str把数值作为类别键。 |
| 16 | `frame[f"km_q{divisor}"] = (commute10 // divisor).astype("int32")` | 固定数据类型；float32节约内存，int类型表达离散分组，str把数值作为类别键。 |
| 17 | `frame["is_30k_spike"] = (income == 30000).astype("int8")` | 固定数据类型；float32节约内存，int类型表达离散分组，str把数值作为类别键。 |
| 18 | `frame["is_millionaire_cliff"] = (income >= 170537).astype("int8")` | 固定数据类型；float32节约内存，int类型表达离散分组，str把数值作为类别键。 |
| 19 | `frame["is_dead_zone"] = ((income >= 38000) & (income <= 42000)).astype("int8")` | 固定数据类型；float32节约内存，int类型表达离散分组，str把数值作为类别键。 |
| 20 | `frame["is_env_hater"] = (frame["Environmental_Concern_Level"] == 1).astype("int8")` | 固定数据类型；float32节约内存，int类型表达离散分组，str把数值作为类别键。 |
| 23 | `frame[column].map(original_means[column]).fillna(original_prior).astype("float32")` | 为未匹配或缺失项提供明确回退值；均值特征通常回退整体购买比例。 |
| 26 | `"k_inc_exact": income.astype(str),` | 固定数据类型；float32节约内存，int类型表达离散分组，str把数值作为类别键。 |
| 27 | `"k_inc100": (income // 100).astype(str),` | 固定数据类型；float32节约内存，int类型表达离散分组，str把数值作为类别键。 |
| 28 | `"k_inc1000": (income // 1000).astype(str),` | 固定数据类型；float32节约内存，int类型表达离散分组，str把数值作为类别键。 |
| 29 | `"k_km_int": (commute10 // 10).astype(str),` | 固定数据类型；float32节约内存，int类型表达离散分组，str把数值作为类别键。 |
| 35 | `keys[f"k_{column}"] = frame[column].astype(str).to_numpy()` | 固定数据类型；float32节约内存，int类型表达离散分组，str把数值作为类别键。 |

## 折内频率、类别与三组目标编码

先构造三份原始特征；频率和类别词表只从训练折学习。auto、10、100三种平滑分别编码同一组key。zip把三份特征与三份编码一一对应；astype float32保持历史精度和内存规模。

In [ ]:
def prepare_hybrid_fold(training_index, validation_index):
    raw_parts = [train.iloc[training_index], train.iloc[validation_index], test]
    built = [make_unsupervised_base(part) for part in raw_parts]
    frames = [item[0] for item in built]
    keys = [item[1] for item in built]
    training_income, training_commute = built[0][2], built[0][3]
    income_frequency = pd.Series(training_income).value_counts()
    commute_frequency = pd.Series(training_commute).value_counts()
    for frame, item in zip(frames, built):
        frame["fq_inc"] = pd.Series(item[2]).map(income_frequency).fillna(0).astype("float32")
        frame["fq_km"] = pd.Series(item[3]).map(commute_frequency).fillna(0).astype("float32")
    for column in keys[0].columns:
        frequency = keys[0][column].value_counts(normalize=True)
        for frame, key_frame in zip(frames, keys):
            frame[f"{column}_fe"] = key_frame[column].map(frequency).fillna(0).astype("float32")
    for column in categorical_columns:
        # 类别词表只从外层训练折学习，未知值成为缺失类别。
        categories = pd.Index(frames[0][column].astype(str).unique())
        for frame in frames:
            frame[column] = pd.Categorical(frame[column].astype(str), categories=categories)
    # 三种平滑提供不同分组统计视角；训练行使用内部交叉拟合。
    for smooth, tag in (("auto", "auto"), (10.0, "10"), (100.0, "100")):
        encoder = TargetEncoder(
            target_type="binary", smooth=smooth, cv=5, shuffle=True, random_state=SEED
        )
        training_encoded = encoder.fit_transform(keys[0], y.iloc[training_index])
        validation_encoded = encoder.transform(keys[1])
        test_encoded = encoder.transform(keys[2])
        for feature_index, column in enumerate(keys[0].columns):
            encoded_name = f"{column}_te{tag}"
            for frame, values in zip(frames, [training_encoded, validation_encoded, test_encoded]):
                frame[encoded_name] = values[:, feature_index].astype("float32")
    assert frames[0].columns.equals(frames[1].columns)
    assert frames[0].columns.equals(frames[2].columns)
    for frame in frames:
        numeric = frame.select_dtypes(exclude="category").to_numpy(dtype=float)
        assert np.isfinite(numeric).all()
    return frames

### 对照上方代码逐句理解

行号从上方代码单元第一行起计。重复操作也列出，方便逐行定位；跨行调用请连同后续参数一起阅读。

| 行 | 代码定位 | 中文解释 |
|---|---|---|
| 7 | `income_frequency = pd.Series(training_income).value_counts()` | 从当前训练部分计算频率；normalize=True返回比例，否则返回次数。 |
| 8 | `commute_frequency = pd.Series(training_commute).value_counts()` | 从当前训练部分计算频率；normalize=True返回比例，否则返回次数。 |
| 10 | `frame["fq_inc"] = pd.Series(item[2]).map(income_frequency).fillna(0).astype("float32")` | 为未匹配或缺失项提供明确回退值；均值特征通常回退整体购买比例。 |
| 11 | `frame["fq_km"] = pd.Series(item[3]).map(commute_frequency).fillna(0).astype("float32")` | 为未匹配或缺失项提供明确回退值；均值特征通常回退整体购买比例。 |
| 13 | `frequency = keys[0][column].value_counts(normalize=True)` | 从当前训练部分计算频率；normalize=True返回比例，否则返回次数。 |
| 15 | `frame[f"{column}_fe"] = key_frame[column].map(frequency).fillna(0).astype("float32")` | 为未匹配或缺失项提供明确回退值；均值特征通常回退整体购买比例。 |
| 18 | `categories = pd.Index(frames[0][column].astype(str).unique())` | 固定数据类型；float32节约内存，int类型表达离散分组，str把数值作为类别键。 |
| 20 | `frame[column] = pd.Categorical(frame[column].astype(str), categories=categories)` | 用训练折词表固定类别编码；未见类别成为缺失，不重新为验证集编号。 |
| 26 | `training_encoded = encoder.fit_transform(keys[0], y.iloc[training_index])` | 训练行使用编码器内部交叉拟合结果，不能替换成fit后对训练集transform。 |
| 27 | `validation_encoded = encoder.transform(keys[1])` | 使用外层训练数据已学到的映射，不输入验证或测试标签。 |
| 28 | `test_encoded = encoder.transform(keys[2])` | 使用外层训练数据已学到的映射，不输入验证或测试标签。 |
| 32 | `frame[encoded_name] = values[:, feature_index].astype("float32")` | 固定数据类型；float32节约内存，int类型表达离散分组，str把数值作为类别键。 |

## 固定历史参数

深度5、32叶、学习率0.02、三组编码。四线程CPU；这里subsample_freq=1，和另一个模型不同，必须保留。

In [ ]:
model_params = {
    "n_estimators": 20000, "learning_rate": 0.02, "max_depth": 5,
    "num_leaves": 32, "min_child_samples": 10, "subsample": 0.8,
    "subsample_freq": 1, "colsample_bytree": 0.3, "reg_alpha": 0.071,
    "reg_lambda": 2.0, "max_bin": 1024, "feature_pre_filter": False,
    "n_jobs": 4, "verbosity": -1, "random_state": SEED,
}

prepare_fold = prepare_hybrid_fold
patience, expected_count = 500, 110

## 五折训练并回填预测

这是主要耗时单元。每折留出五分之一用于评价；早停也使用这个验证集，因此OOF属于开发评价。predict_proba的[:,1]取购买概率。五次覆盖完成后每行coverage应为1。不要为了整理日志重训。

In [ ]:
run_name = 'hybrid'
method_sources = ['https://www.kaggle.com/code/megayak/s6e9-one-lightgbm-from-raw-data-cv-0-9463']

candidate_oof = np.full(len(train),np.nan)  # 没有预测的行保持NaN，便于发现漏填。
candidate_test = np.zeros(len(test))
coverage = np.zeros(len(train),dtype=np.uint8)
records, fold_features = [], {}
started = perf_counter()
for fold in range(N_SPLITS):
    training_index = np.flatnonzero(fold_ids != fold)
    validation_index = np.flatnonzero(fold_ids == fold)
    X_train, X_valid, X_test = prepare_fold(training_index,validation_index)
    assert X_train.columns.equals(X_valid.columns) and X_train.columns.equals(X_test.columns)
    if expected_count is not None:
        assert X_train.shape[1] == expected_count, (fold,X_train.shape)
        historical_columns = foundation_note.get('historical_features',{}).get(run_name,{}).get(str(fold))
        if historical_columns is not None:
            assert X_train.columns.tolist() == historical_columns, 'Historical feature order differs.'
    fold_features[str(fold)] = X_train.columns.tolist()
    if run_name != 'baseline':
        assert model_params == foundation_note['historical_params'][run_name], 'Historical parameters differ.'
    model = lgb.LGBMClassifier(**model_params)  # 每个外层fold创建全新模型。
    fold_started = perf_counter()
    model.fit(X_train,y.iloc[training_index],eval_set=[(X_valid,y.iloc[validation_index])],
              eval_metric='auc',callbacks=[lgb.early_stopping(patience,first_metric_only=True,verbose=False),lgb.log_evaluation(1000)])
    probability = model.predict_proba(X_valid,num_iteration=model.best_iteration_)[:,1]
    test_probability = model.predict_proba(X_test,num_iteration=model.best_iteration_)[:,1]
    assert np.isfinite(probability).all() and np.isfinite(test_probability).all()
    candidate_oof[validation_index] = probability  # 回填官方训练行的位置。
    coverage[validation_index] += 1
    candidate_test += test_probability/N_SPLITS  # 在概率空间平均，暂不排名。
    records.append({'fold':fold,'auc':float(roc_auc_score(y.iloc[validation_index],probability)),
                    'features':X_train.shape[1],'best_iteration':int(model.best_iteration_),
                    'fit_seconds':perf_counter()-fold_started})
    print(records[-1])
    del model,X_train,X_valid,X_test
    gc.collect()
assert (coverage==1).all() and np.isfinite(candidate_oof).all()
assert ((candidate_oof>=0)&(candidate_oof<=1)).all()
assert ((candidate_test>=0)&(candidate_test<=1)).all()
elapsed = perf_counter()-started
display(pd.DataFrame(records))
print('Overall OOF AUC:',roc_auc_score(y,candidate_oof))

## 保存并交接真实结果

只保存完整OOF、测试预测和摘要。输出目录已经存在时停止，先保存上次结果后重开会话。把整个目录保存为Notebook输出，告诉融合负责人挂载。

In [ ]:
output = Path('/kaggle/working')/run_name
output.mkdir(exist_ok=False)
oof = pd.DataFrame({'id':train.id,'target':y,'fold':fold_ids,'prediction':candidate_oof})
submission = sample.copy()
submission[TARGET] = candidate_test
auc = float(roc_auc_score(y,candidate_oof))
report = (f'The {run_name} model was trained on five shared folds. Overall OOF ROC AUC was {auc:.9f}. '
          'Test probabilities were averaged across five models. All competition-derived supervised '
          'preprocessing was fitted within outer training folds. These are development results; '
          'leaderboard performance for this run remains unverified.')
summary = {'run_name':run_name,'model_params':model_params,'stopping_rounds':patience,
           'fold_features':fold_features,'fold_metrics':records,'oof_auc':auc,'elapsed_seconds':elapsed,
           'versions':current_versions(),'fold_source':str(fold_path),'method_sources':method_sources,
           'test_aggregation':'mean probabilities across five folds','public_score':None,'report_summary':report}
oof.to_csv(output/'oof_predictions.csv',index=False)
submission.to_csv(output/'submission.csv',index=False)
(output/'run_summary.json').write_text(json.dumps(summary,indent=2,allow_nan=False),encoding='utf-8')
print(report)
print('Saved:',output)

## 中文结果解析与理解检查

逐折AUC比较必须使用相同fold。整体OOF AUC和五折AUC的平均不是同一个量。两个完整方案同时改变多组特征，不能宣称某一列造成全部提升。

请回答：为什么测试集不参与早停？为什么目标编码训练行不能直接使用全训练折groupby均值？为什么相同特征数量仍可能有不同列序？请画出一行样本从原始数据到验证预测经过的步骤。

运行后用实际输出写中文观察；摘要已自动生成英文Report Summary。历史参照：收入邻域模型0.946046626，混合特征模型0.946129123；未训练前不能把这些数写成本次结果。